# 09 Equity Analysis

## Purpose
This notebook combines route ridership data with scheduled service frequency data to evaluate route productivity. The resulting dataset serves as the foundation for identifying corridors that may warrant additional transit investment.

## Inputs
- route_ridership.csv
- route_frequency.csv

## Outputs
- route_investment_metrics.csv

## Why This Matters
Ridership alone does not indicate whether a route is underserved. By comparing ridership against the amount of service provided, we can identify routes carrying large numbers of riders relative to their scheduled trips.


## 1. Load Ridership and Service Frequency Data

In [5]:
import pandas as pd

ridership = pd.read_csv(
    "../data/processed/route_ridership.csv"
)

frequency = pd.read_csv(
    "../data/processed/route_frequency.csv"
)

print(ridership.shape)
print(frequency.shape)

ridership.head()

(5, 10)
(116, 10)


,route_long_name,route_type,route_text_color,route_color,agency_id,route_id,route_url,route_desc,route_short_name,avg_weekday_boardings
0,Richmond,3,FFFFFF,4080,HOU,25,https://assets-cdn-prod.azureedge.us/assets/do...,NaN,25,7489
1,Gessner,3,FFFFFF,4080,HOU,46,https://assets-cdn-prod.azureedge.us/assets/do...,NaN,46,7670
2,Westheimer,3,FFFFFF,4080,HOU,82,https://assets-cdn-prod.azureedge.us/assets/do...,NaN,82,14054
3,Bellaire,3,FFFFFF,4080,HOU,2,https://assets-cdn-prod.azureedge.us/assets/do...,NaN,2,7590
4,Beechnut,3,FFFFFF,4080,HOU,4,https://assets-cdn-prod.azureedge.us/assets/do...,NaN,4,8743


## 2. Verify Available Columns
Inspect the datasets before merging to ensure the required route identifiers and metrics are present.

In [6]:
print(ridership.columns.tolist())
print(frequency.columns.tolist())

['route_long_name', 'route_type', 'route_text_color', 'route_color', 'agency_id', 'route_id', 'route_url', 'route_desc', 'route_short_name', 'avg_weekday_boardings']
['route_long_name', 'route_type', 'route_text_color', 'route_color', 'agency_id', 'route_id', 'route_url', 'route_desc', 'route_short_name', 'scheduled_trips']


## 3. Merge Datasets and Calculate Productivity
Routes are merged using route_id. Riders per trip is calculated as average weekday boardings divided by scheduled trips.

In [7]:
master = ridership.merge(
    frequency[["route_id", "scheduled_trips"]],
    on="route_id",
    how="inner"
)

master["riders_per_trip"] = (
    master["avg_weekday_boardings"] / master["scheduled_trips"]
)

master[
    [
        "route_id",
        "route_long_name",
        "avg_weekday_boardings",
        "scheduled_trips",
        "riders_per_trip"
    ]
].sort_values(
    "riders_per_trip",
    ascending=False
)

,route_id,route_long_name,avg_weekday_boardings,scheduled_trips,riders_per_trip
4,4,Beechnut,8743,896.0,9.757812
2,82,Westheimer,14054,1487.0,9.451244
1,46,Gessner,7670,854.0,8.981265
0,25,Richmond,7489,942.0,7.950106
3,2,Bellaire,7590,1013.0,7.492596


## 4. Save Processed Dataset
Export the combined dataset for use in investment-priority analysis and final recommendations.

In [8]:
master.to_csv(
    "../data/processed/route_investment_metrics.csv",
    index=False
)

print("saved")

saved


## Summary

This notebook created a route-level performance dataset combining ridership and service supply. The resulting metrics help identify corridors that may have strong demand relative to their current level of service.
